# <b>ADM 3080 — Analítica con Python — Universidad San Francisco de Quito</b>
<h2><b>Proyecto de Prueba Técnica</b></h2>
<i>Viernes, 24 de abril de 2026</i>
<hr>
<ul>
    <li><b>Integrantes del Grupo</b>: Santiago Arellano [00328370]</li>
    <li><b>NRC del Curso</b>: 2041</li>
    <li><b>Profesor</b>: Juan Felipe Nájera Puente</li>
</ul>
<hr>

## Sección I: Introducción al Proyecto

En esta sección se presentará una visión general del proyecto, analizando la base de datos y definiendo métricas de calidad para los datos, orientadose en la definición de límites por métricas faltantes e.g. límites porcentuales tolerables de anclajes con inicio pero sin registro de terminación, etc. Además se analizará el tipo de dato esperado por cada una de las clases y se establecerán criterios para la limpieza de datos, incluyendo la identificación y manejo de valores atípicos, datos faltantes y errores de formato. Se discutirá la importancia de la calidad de los datos en el contexto del análisis y cómo una limpieza adecuada puede mejorar la precisión y confiabilidad de los resultados obtenidos a partir de los datos.

### Contexto del Dataset y el Problema

El dataset representa datos registrados de una aplicación de Wellness desarrollada por Yana, una empresa mexicana orientada a ofrecer un servicio de acompañamiento a individuos para desarrollar su mejor versión. Los datos recopilados atienden a una sola sección de la aplicación, el registro de *anclajes* correspondientes a actividades de bienestar como **volver al presente (Present)** o **distraer mi mente (Mind)** registradas junto con la fecha y hora de inicio y terminación de la actividad. Además, el dataset contiene información correspondiente al proceso, sea iniciar o terminar una actividad de anclaje y datos anonimizados de los usuarios.

Debido a las leyes de protección de datos generales, la empresa ha optado por la anonimización de los datos, usando un cifrado, muy probablemente SHA256 o alguna variante interna de tipo *one way* para reducir la posibilidad de identificación de los usuarios. El tipo de cifrado además, dificulta la agregación de datos mediante IDS, por lo que no podemos asumir la posibilidad de identificar el proceso de cerrado y apertura de una actividad en base a un ID específico, dado que los IDs de inicio y terminación de una actividad no necesariamente coincidirán, lo que complica la identificación de actividades completas. Esto se traduce en la necesidad de eliminar esta columna del análisis, tanto el ID registrado bajo la columna de &lt;anonymous&gt; junto con el id registrado en la columna de IDs no son necesarios para un análisis de datos de agregación, no datos por usuario.

En este contexto, la gerencia ha definido diferentes preguntas de análisis que se usarán para orientar diferentes niveles de marketing, UX design y otras áreas afines a esta aplicación y *customer journey* planteado para mejorar la experiencia de usuario.

<hr>

### Preguntas de Análisis expandidas
1. ¿Cuál es la frecuencia de uso de cada técnica de anclaje?
    <blockquote>En este caso, la frecuencia de uso se traduce directamente a la cantidad de sesiones por tecnica de anclaje iniciada, lo que se puede medir de dos formas distintas, agregadas sobre el dataset (visión general para gerencia), o una visión análitica, con un gráfico de área organizado por día de la semana, més o año. En este caso, el dataset proporcionado agrega información diaria, lo que nos permitiría visualizar la tendencia de uso por día de la semana junto con día del mes para determinar la relación entre día de la semana y actividad.
    <br><br>
    Asimismo, el nivel de agregación por día se puede usar para medir mediante agregaciones al 100% la distribución porcentual de las actividades registradas, lo que permitiría identificar la técnica de anclaje más utilizada por los usuarios, lo que a su vez podría ser útil para orientar campañas de marketing o mejorar la experiencia de usuario en torno a esta técnica.
    <br><br>
    En este contexto, entonces, hay tres diferentes formas de evaluar esta frecuencia, la primera es una agregación totalizada, que permite ver durante todo el periodo de análisis los resultados de las actividades, la segunda es una visualización por día mediante líneas, mostrando la suma totalizada de actividades, y la tercera es una distribución porcentual por día, más visual, más facil de entender para personas no técnicas y con poco tiempo para revisar visualizaciones complejas.
    </blockquote>
2. ¿Qué porcentaje de usuarios finaliza cada tipo de ejercicio?
    <blockquote>
    Para esta pregunta, la información importante es la visión desglosada por día y por tipo de actividad. Cuando hablamos de porcentaje de usuarios finalizados, debemos de tomar en cuenta que para cada tipo de actividad existe la posibilidad que un usuario entre, y nunca salga, o entre y salga, o entre y salga de la aplicación y no termine la actividad, por lo que tenemos que comparar la cantidad de registros de inicio y fin de actividades. De nuevo esta pregunta tiene dos niveles diferentes de agregación. El nivel general, del dataset completo, corresponde a la totalización de eventos cerrados y abiertos y la diferencia porcentual para evaluar si existen deficiencias entre la apertura de algunos tipos de actividades comparados con otros, lo que podría indicar problemas de experiencia de usuario o falta de interés en ciertas actividades.
     <br><br>
     El segundo nivel de análisis corresponde a la evaluación por día, lo que permitiría identificar tendencias específicas en ciertos días de la semana o meses, lo que podría ser útil para orientar campañas de marketing o mejorar la experiencia de usuario en torno a estas técnicas. Para la primera visualización, una visualización en torno a barras apiladas al 100% sería ideal para mostrar la distribución porcentual a nivel de actividades, mientras que para la visión diaria, una visualización de líneas mostrando la tendencia de finalización por día sería ideal para mostrar la evolución de esta métrica a lo largo del tiempo.
     <br><br>
     Una tercera visualización efectiva corresponde a usar un diagrama de tipo sankey donde todos los usuarios registrados que han iniciado se distribuyan a la terminación de sus eventos, este tipo de gráfico es fuerte para una visualización de transición en el flujo y es intuitivo dado que mostraría la diferencia entre el ingreso de usuarios al iniciar la actividad y la baja cantidad de usuarios posibles al terminar la misma.
    </blockquote>
3. ¿Qué técnicas suelen utilizarse juntas en una misma sesión o en momentos cercanos?
    <blockquote>
    Para esta pregunta, es importate definir primero que significa durante una misma sesión, o durante momentos cercanos. Dado que no tenemos una información accionable del usuario, dado que no tenemos IDs agrupables y no tenemos forma de regresar de la encriptación de los PII de los usuarios, no podemos analizar en general mediante usuarios. Por otro lado, si bien tenemos inforamción de fecha y hora, no existe un registro que indique que estos eventos fueron parte de una misma sesión, por lo que para esta pregunta vamos a asumir que **una sesión dentro de la aplicación puede dura una hora o más** que sería nuestro indicador de que si dentro de una misma sesión (en el rango de 1 hora) se inician más de una actividad entonces tenemos una sesión y podemos evaluar la cantidad de eventos que se realizan juntos.
    <br><br>
    Para evaluar eventos que se realizen en eventos cercanos, podemos usar una noción de que un evento cercano corresponde a eventos **que se inician dentro de 30 minutos** entre sí, lo que nos permitiría centrar este concepto de momentos cercanos y analizar que eventos se inician seguidos en un periodo de 30 minutos de análisis luego de una actividad iniciada.
    </blockquote>

<hr>

### Guía base de la estructura del dataset

Los datos están distribuidos en un solo archivo *CSV*, importando desde *GitHub Raw links* para mayor comodidad a la hora de ejecutar este cuaderno de Python en otros sistemas en donde el acceso a archivos csv, la forma de manejar archivos del sistema operativo o la ubicación del archivo podrían causar problemas de ejecución innecesarios. En base de este archivo *CSV* y la guía del dataset se ha definido el siguiente listado de los campos y su respectiva descripción.

 <div style="display: flex; align-items:center; align-self:center; justify-content:center">
  <table style="margin: auto; border-collapse: collapse; width: 80%; text-align: center; border: 1px solid black;">
      <thead>
          <tr style="background-color: #f2f2f2;">
              <th style="border: 1px solid black; padding: 8px;">Campo</th>
              <th style="border: 1px solid black; padding: 8px;">Tipo de Dato Esperado</th>
              <th style="border: 1px solid black; padding: 8px;">Descripción</th>
          </tr>
      </thead>
      <tbody>
          <tr>
              <td style="border: 1px solid black; padding: 8px;">id</td>
              <td style="border: 1px solid black; padding: 8px;">Entero</td>
              <td style="border: 1px solid black; padding: 8px;">ID de identificación única del registro</td>
          </tr>
          <tr>
              <td style="border: 1px solid black; padding: 8px;">event</td>
              <td style="border: 1px solid black; padding: 8px;">Texto</td>
              <td style="border: 1px solid black; padding: 8px;">Tipo de evento registrado</td>
          </tr>
          <tr>
              <td style="border: 1px solid black; padding: 8px;">techniqueID</td>
              <td style="border: 1px solid black; padding: 8px;">Texto</td>
              <td style="border: 1px solid black; padding: 8px;">ID de la técnica utilizada</td>
          </tr>
          <tr>
              <td style="border: 1px solid black; padding: 8px;">date</td>
              <td style="border: 1px solid black; padding: 8px;">Fecha</td>
              <td style="border: 1px solid black; padding: 8px;">Fecha en que ocurrió el evento</td>
          </tr>
      </tbody>
  </table>
  </div>

En el contexto del trabajo de análisis a desarrollar, notamos que la parte más importante a considerar es el desarrollo de métricas de calidad para las columnas de evento, tipo de técnica y fecha, dado que en base a estas se construye todo el análisis base de las preguntas definidas para la prueba.

<hr>

### Consideraciones de Limpieza por Columna

No se considera necesario la modificación de la columna de ID (será eliminada al no ser útil para la identificación de patrones de usuarios o de actividades), y tampoco para techniqueID dado que los datos registrados se encuentran definidos en base a las mismas clases, lo que garantiza que no existan datos incorrectos en esta clase.

#### <code>event</code>

<i><small>Esta columna, que define el tipo de evento disparado desde la UI de la aplicación para el módulo de actividades de anclaje puede tomar dos valores específicos: <code>{"Grounding Started","Grounding Ended"}</code> que corresponden al inicio y fin de una actividad de anclaje que registra los datos de esta en las columnas adicionales de date y techniqueID.</small></i>

El principal problema de event es que no sabemos si los eventos se disparan solo por **terminar una sesión** o si también se disparan **automáticamente cuando una sesión no se termina por mucho tiempo**, o si por ejemplo se disparan cuando el **usuario sale de la aplicación durante una sesión**. Esta ambigüedad de la razón de terminación es un problema considerable dado que:
1. Si la sesión marcó **Grounding Ended** debido a un timeout en una actividad, no se la puede considerar como cerrada y por ejemplo esto sería una clase faltante en los datos, que sería una clase de tipo *Grounding Timeout* que de estar presente nos permitiría identificar una tercera clase de terminación útil para un análisis adicional, lo que puede ser una sugerencia de mejora para el equipo de sistemas con respecto a los datos recopilados por la aplicación.
2. Si la sesión marcó **Grounding Ended** debido a una salida de la aplicación por el usuario entonces es posible que tengamos multiples registros de *Grounding Ended* seguidos si por ejemplo el usuario tiene que responder a llamadas, mensajes, o atender a otros requerimientos de sus vidas por fuera de la aplicación. Esta puede ser una razón por la que existen varios registros seguidos, que varían por minutos en algunos casos, de terminaciones.
3. Si la sesión marcó **Grounding Ended** debido a una terminación normal, entonces no existe un problema de calidad de datos, dado que el evento se disparó correctamente y se puede usar para el análisis.

Dado que una visión general de los datos basados en el archivo .csv propuesto, muestra que los primeros datos registran un evento de **Grounding Start** y luego cinco diferentes eventos de cierre a la misma hora, con una duración de 1 minuto luego de una apertura de proceso, esto nos muestra que puede existir un problema de calidad de datos con respecto al segundo escenario de **Grouding Ended**.

<b><small>En este caso, aunque no podemos inferir que tipo de datos se registran en los registros duplicados, podemos tomar dos rutas diferentes de análisis, eliminando entradas duplicadas en base del tipo de evento, acción y hora, lo que nos podría ayudar a reducir el problema de duplicados, otra alternativa es notar su existencia pero mantenerlos en el dataset para argumentar sobre ellos luego</small></b>

#### <code>date</code>

<hr style="height:1px;border:none;color:#333;background-color:#333;">

<i><small>Esta columna registra la fecha y hora exacta de inicio y fin de la actividad. La fecha se registra en un formato de mm/dd/yyyy formato que es similar al usado en EEUU, lo que nos permite una conversión directa en terminos de fecha para su análisis posterior. Además el dataset registra la fecha en formato hh:mm con formato de 24h en lugar de solo 12, lo que nos indica un formato de datos mucho más estandarizado enfocado en la utilidad de los datos bajo sistemas estandarizados para formatos en-US y hora basado en 24.</small></i>

En este caso, en base a la columna de fecha y hora podemos realizar los siguientes pasos de análisis:

1. Primero, **dividir la columna de date en dos, date y time** para poder realizar análisis de datos diferenciados sin tener que realizar una modificación *inline* a la columna de datos durante el análisis.
2. Segundo, **derivar día de la semana, nombre del día, mes, año y nombre del mes** en base de la columna de *date* para extraer datos para agregaciones de tipo timeseries en base a fechas en primera instancia y para agrupaciones por día de la semana o nombre del mes.
3. Tercero, **en base a nuestros periodos base de longitud de actividad y eventos cercanos** podemos derivar columnas de tipo *session* y *eventos cercanos* para poder realizar análisis de tipo sesión y eventos cercanos, lo que nos permitiría responder a la pregunta número 3 de nuestro análisis.

<b><small>En este caso, no existe una remediación de datos necesaria pero si una derivación de columnas adicionales para un análisis profundo de datos.</small></b>

### Librerías Requeridas para el Análisis

Para el presente proyecto, se ha optado por trabajar mediante un entorno *managed* de Python a través de *Python Venv(s)* permitiendo el aislamiento de las librerías requeridas para el trabajo. En este contexto, los comandos a ejecutar a continuación se encargan de la instalación de varias librerías para el trabajo, lo que requiere que Python se encuentre instalado y definido dentro del **PATH** o en su caso, en la carpeta de configuración */bin* o */sbin*.
<br><br>
Para la realización del trabajo, se ha considerado necesario la inclusión de diversas librerías. El siguiente listado detalla la razón de la inclusión de la librería en el desarrollo del proyecto:

<div style="display: flex; align-items:center; align-self:center; justify-content:center">
<table style="margin: auto; border-collapse: collapse; width: 80%; text-align: justify; border: 1px solid black;">
    <thead>
        <tr style="background-color: #f2f2f2;">
            <th style="border: 1px solid black; padding: 8px;">Librería</th>
            <th style="border: 1px solid black; padding: 8px;">Uso</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border: 1px solid black; padding: 8px;">Numpy</td>
            <td style="border: 1px solid black; padding: 8px;">Librería de análisis numérico con fuertes características para operaciones algebráicas, matriciales y con un backend implementado en C++ por lo que permitirá la realización de operaciones numéricas rápidamente.</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 8px;">Pandas</td>
            <td style="border: 1px solid black; padding: 8px;">Librería base para el análisis del archivo CSV, su manipulación en forma de carga, datos, y agregaciones que permitirán dar respuesta a las preguntas de investigación definidas.</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 8px;">Matplotlib</td>
            <td style="border: 1px solid black; padding: 8px;">Base de las visualizaciones en el caso de ser necesarias para el desarrollo del proyecto o la exploración visual de los datos.</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 8px;">Seaborn</td>
            <td style="border: 1px solid black; padding: 8px;">Apoyo a Matplotlib en el caso de requerir visualizaciones más avanzadas.</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 8px;">Squarify</td>
            <td style="border: 1px solid black; padding: 8px;">Apoyo a Matplotlib para la realización de Treemaps en Python.</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 8px;">SciPy</td>
            <td style="border: 1px solid black; padding: 8px;">Para pruebas de hipótesis estadísticas (t-test, Mann-Whitney, etc.).</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 8px;">WordCloud</td>
            <td style="border: 1px solid black; padding: 8px;">Para mapas de palabras como visualizacion de peso por clase.</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 8px;">Plotly</td>
            <td style="border: 1px solid black; padding: 8px;">Para gráficos adicionales.</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 8px;">MissingNo</td>
            <td style="border: 1px solid black; padding: 8px;">Para gráficos adicionales especializados en al visualización de valores nulos.</td>
        </tr>
    </tbody>
</table>
</div>
<br><br>
En este contexto, las librerías seran importadas  usando un álias al nombre original para facilidad de uso durante el desarrollo del proyecto.

In [2]:
#? 1. Importacion de librerias para el proyecto
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import squarify as sqs
import plotly as plty

## Sección II: Análisis de Calidad de los Datos Iniciales

En esta sección se realizará un análisis inicial de los datos, revisando distribución de valores mediante agrupaciones, conteos, visualizaciones de peso mediante wordcloud, y conteo de nulos. Para valores de tipo fecha se realizará un análisis en base a una corrección inline (previa a la definición del dataset procesado) para analizar la distribución de fechas y de horas visualmente.


## Sección III: Preprocesamiento de Datos Iniciales

En esta sección se trabajará para aplicar las medidas de limpieza de datos definidas en la sección I, aplicando las correcciones necesarias para el análisis, como la eliminación de columnas no útiles, la derivación de columnas adicionales para el análisis, y la corrección de valores atípicos o nulos en base a las métricas definidas en la sección I.

## Sección IV: Análisis de Datos

En esta sección se trabajará con un análisis de datos en base de agrupaciones y visualizaciones para identificar patrones y responder a las preguntas de negocio. Dado que no tenemos una cantidad significativa de datos numéricos para realizar un análisis estadísticos, se considerará, con vista a los resultados de agregaciones y visualizaciones la posibilidad de realizar pruebas de hipótesis estadísticas para evaluar la significancia de los resultados obtenidos, lo que podría ser útil para identificar patrones significativos en los datos y evitar conclusiones erróneas basadas en patrones aleatorios o ruido en los datos.